# 01 · CFTR2 — the disease-specific *functional* truth set

[CFTR2](https://cftr2.org) classifies *CFTR* variants using **patient outcomes + in-vitro CFTR function assays** — a different, more *functional* kind of evidence than ClinVar's clinical assertions. This notebook builds the current CFTR2 release locally from the published workbook (gitignored), then reconstructs when each variant's CF-causing call was made by chaining CFTR2's twelve published releases.

CFTR2 is **not** independent of ClinVar — the two share clinical evidence and cross-cite. What that costs a benchmark is worked through in [`../tools/05_revel.ipynb`](../tools/05_revel.ipynb) §2 rather than repeated here.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · CFTR2 — a disease-specific, *functional* reference

[CFTR2](https://cftr2.org) is different in kind from ClinVar. It is a **CF-specific** database that classifies *CFTR* variants as:

- **CF-causing**
- **Varying clinical consequence** (formerly "CF-causing (mild)")
- **Non CF-causing**
- **No interpretation available** — not yet enough evidence

Crucially, CFTR2's calls are built from **two kinds of evidence together**:
1. **Patient data** — real clinical outcomes across thousands of people with CF who carry the variant.
2. **In-vitro CFTR function assays** — measuring, in the lab, how much working chloride-channel the variant protein actually produces.

That second, functional axis is a wet-lab signal no sequence model trained on. It does not make CFTR2 an independent gold standard — see [`../tools/05_revel.ipynb`](../tools/05_revel.ipynb) §2.

### Building the data — a download, no API

CFTR2 has no API — the variant list is published as an Excel workbook, linked from
the site's **CFTR2 Variant List History** page. The workbooks are served from a stable
public path:

```
https://cftr2.org/sites/default/files/CFTR2_<DDMonthYYYY>.xlsx
```

Set `CFTR2_XLSX_NAME` below to the release you want and put it in `data/`
(gitignored — never commit it). Section 2 downloads the whole series automatically;
this cell uses whichever single release you point it at.

**Version control.** Past releases stay available, so a run *can* be pinned to a specific
release rather than to "whatever was current that day". What the cell below does:

- Reads the release date straight out of the workbook's own header metadata
  (the `Date:` row CFTR2 ships in every release) rather than assuming one.
- Persists that date into `data/cftr2_cftr.release.json` alongside the extract,
  so `load_cftr2()` can expose it as a `cftr2_release` column.
- To reproduce a past run, point `CFTR2_XLSX_NAME` at that release's workbook — the
  recorded date reflects that file's own header, not today's date.

The cell reads two sheets from the workbook: **"CFTR2 variants by legacy
name"** (the variant list itself — legacy name, protein name, cDNA name, allele
count/frequency, and functional class) and **"Genomic coordinates"** (authoritative
GRCh38 positions). It derives the 1-letter `protein_variant` key for simple
single-residue missense variants only (regex on the protein name), resolves a
handful of variants listed under a **pipe-combined cDNA name** (e.g. W1282X is
`c.3845G>A|c.3846G>A`, two SNVs that create the same stop codon) by trying each
`|`-separated alternative against the genomic sheet, and writes
`data/cftr2_cftr.csv`. It also asserts the workbook's header states the
expected MANE transcript (`NM_000492.4`) before trusting its coordinates.

License: CFTR2's terms permit downloading for your own non-commercial use but forbid
republishing any portion of the Content — so the workbooks and every extract built from
them stay local. Cite CFTR2 if you use it; see `data_manifest.json`.

In [2]:
import re, json, openpyxl
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
# Change this if you've manually sourced a different (e.g. older) CFTR2 release --
# whatever release date IS in that file's own header is what gets recorded.
CFTR2_XLSX_NAME = "CFTR2_30January2026.xlsx"
CFTR2_XLSX = DATA_DIR / CFTR2_XLSX_NAME
CFTR2_TSV = DATA_DIR / "cftr2_cftr.csv"
CFTR2_RELEASE_JSON = DATA_DIR / "cftr2_cftr.release.json"
EXPECT_TX = "NM_000492.4"
AA3TO1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q",
    "Glu": "E", "Gly": "G", "His": "H", "Ile": "I", "Leu": "L", "Lys": "K",
    "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S", "Thr": "T", "Trp": "W",
    "Tyr": "Y", "Val": "V",
}
MIS = re.compile(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")   # simple single-residue missense only


def missense_key(protein_name: str) -> str:
    if not protein_name:
        return ""
    m = MIS.match(protein_name.strip())
    if not m:
        return ""
    a, pos, b = m.group(1), m.group(2), m.group(3)
    return f"{AA3TO1[a]}{pos}{AA3TO1[b]}" if a in AA3TO1 and b in AA3TO1 else ""


if CFTR2_TSV.exists():
    print(f"already built -> {CFTR2_TSV.name} (delete it and {CFTR2_RELEASE_JSON.name} to rebuild)")
elif not CFTR2_XLSX.exists():
    raise FileNotFoundError(
        f"{CFTR2_XLSX} not found.\n"
        "CFTR2 has no API -- get the variant-list workbook:\n"
        "  1. Go to https://cftr2.org and download the current release xlsx\n"
        f"  2. Save it as {CFTR2_XLSX} (do NOT commit it -- data/ is gitignored)\n"
        "     (or set CFTR2_XLSX_NAME above to whatever you saved it as)\n"
        "Then re-run this cell."
    )
else:
    wb = openpyxl.load_workbook(CFTR2_XLSX, read_only=True, data_only=True)
    ws = wb["CFTR2 variants by legacy name"]

    # Header rows (1-12) carry provenance: release date, official counts, and --
    # critically -- the reference transcript, so the build documents its own basis
    # instead of assuming GRCh38/MANE.
    header_meta = {}
    for r in ws.iter_rows(min_row=1, max_row=12, values_only=True):
        cell = str(r[0]).strip() if r[0] is not None else ""
        if ":" in cell:
            k, v = cell.split(":", 1)
            header_meta[k.strip()] = v.strip()
    tx = header_meta.get("CFTR reference transcript", "")
    assert EXPECT_TX in tx, (
        f"CFTR2 header transcript is {tx!r}, expected {EXPECT_TX}; the extract's "
        "genomic coordinates + MANE assumptions may no longer hold -- check the release.")
    print("CFTR2 header provenance:")
    for k in ("Date", "Number of patients in CFTR2", "Number of variants reported in CFTR2",
              "Number of variants with interpretations", "CFTR reference transcript"):
        if k in header_meta:
            print(f"  {k}: {header_meta[k]}")

    rows = []
    for r in ws.iter_rows(min_row=13, values_only=True):
        if r[0] is None:
            continue
        legacy, protein, cdna, alt, alleles, af, prev, cur, changed = r[:9]
        rows.append({"protein_variant": missense_key(protein or ""), "legacy_name": legacy,
                     "protein_name": protein, "cdna_name": cdna, "cftr2_alleles": alleles,
                     "cftr2_af": af, "cftr2_class": cur})
    df = pd.DataFrame(rows)

    # Merge GRCh38 genomic coordinates from sheet 2 (on cDNA name).
    ws2 = wb["Genomic coordinates"]
    g = pd.DataFrame(ws2.iter_rows(min_row=2, values_only=True),
                      columns=list(next(ws2.iter_rows(min_row=1, max_row=1, values_only=True))))
    gcols = {"Variant cDNA name": "cdna_name", "grch38_chr": "grch38_chr",
             "grch38_pos": "grch38_pos", "grch38_ref": "grch38_ref", "grch38_alt": "grch38_alt"}
    g = g[list(gcols)].rename(columns=gcols).drop_duplicates("cdna_name")
    gcoord = {k: v for k, v in g.set_index("cdna_name")
              [["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"]].to_dict("index").items()}

    def resolve_coords(cdna):
        # Some CFTR2 variants are listed under a PIPE-combined cDNA name (e.g. W1282X is
        # 'c.3845G>A|c.3846G>A') but the genomic sheet keys each single name separately.
        # Try the whole name, then each alternative, taking the first with coordinates.
        if not isinstance(cdna, str):
            return {}
        for alt in [cdna, *cdna.split("|")]:
            hit = gcoord.get(alt.strip())
            if hit and pd.notna(hit.get("grch38_pos")):
                return hit
        return {}

    coords = pd.DataFrame([resolve_coords(c) for c in df["cdna_name"]], index=df.index,
                           columns=["grch38_chr", "grch38_pos", "grch38_ref", "grch38_alt"])
    df = pd.concat([df, coords], axis=1)
    df.to_csv(CFTR2_TSV, index=False)

    CFTR2_RELEASE_JSON.write_text(json.dumps({
        "release_date": header_meta.get("Date", "unknown"),
        "source_xlsx": CFTR2_XLSX_NAME,
        "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "row_count": len(df),
    }, indent=2))

    print(f"\nwrote {CFTR2_TSV.relative_to(DATA_DIR.parent)} and {CFTR2_RELEASE_JSON.name}  rows: {len(df):,}")
    print("with a 1-letter missense key:", (df["protein_variant"] != "").sum(),
          f"(of {len(df)}; the rest are non-missense -- join by cdna_name/genomic coords)")

already built -> cftr2_cftr.csv (delete it and cftr2_cftr.release.json to rebuild)


In [3]:
cftr2 = tk.load_cftr2()          # the built extract, once you've run the build cell above
print('source :', cftr2['source'].unique(), '| release:', cftr2['cftr2_release'].iloc[0])
print('variants:', len(cftr2), '| with a missense key:', (cftr2['protein_variant'] != '').sum())
print()
print(cftr2['cftr2_class'].value_counts().to_string())
cftr2.head(8)

source : ['REAL'] | release: 30 January 2026
variants: 2097 | with a missense key: 780

cftr2_class
CF-causing                      1245
No interpretation available      722
Varying clinical consequence      83
Non CF-causing                    42


,protein_variant,legacy_name,protein_name,cdna_name,cftr2_alleles,cftr2_af,cftr2_class,grch38_chr,grch38_pos,grch38_ref,grch38_alt,cftr2_release,source
0,,F508del,p.Phe508del,c.1521_1523del,137363,0.650682595473364,CF-causing,7.0,117559590.0,ATCT,A,30 January 2026,REAL
1,,G542X,p.Gly542X,c.1624G>T,5752,0.027246975453089916,CF-causing,7.0,117587778.0,G,T,30 January 2026,REAL
2,G551D,G551D,p.Gly551Asp,c.1652G>A,3831,0.01814728146049852,CF-causing,7.0,117587806.0,G,A,30 January 2026,REAL
3,N1303K,N1303K,p.Asn1303Lys,c.3909C>G,3551,0.01682093355944407,CF-causing,7.0,117652877.0,C,G,30 January 2026,REAL
4,,W1282X,p.Trp1282X,c.3845G>A|c.3846G>A,2500,0.011842391973700416,CF-causing,7.0,117642565.0,G,A,30 January 2026,REAL
5,R117H,R117H,p.Arg117His,c.350G>A,2262,0.010714996257804137,Varying clinical consequence,7.0,117530975.0,G,A,30 January 2026,REAL
6,,3849+10kbC->T,p.?,c.3718-2477C>T,1990,0.00942654401106553,CF-causing,7.0,117639961.0,C,T,30 January 2026,REAL
7,,621+1G->T,p.?,c.489+1G>T,1860,0.008810739628433109,CF-causing,7.0,117531115.0,G,T,30 January 2026,REAL


### Which variants get a `protein_variant` key — and what are the rest?

The 1-letter `protein_variant` key (e.g. `G551D`) is derived by the build cell above from
the protein name with a regex for *simple single-residue missense* only. It exists for
~780 of ~2,097 variants. **The other ~1,317 are NOT all splice variants** — most are
deletions and nonsense. They carry an empty key and must be joined by `cdna_name` or
genomic coordinates instead.

In [4]:
import re
cf = cftr2.copy()
cf["has_key"] = cf["protein_variant"].fillna("") != ""
print("with missense key:", int(cf["has_key"].sum()), "| without:", int((~cf["has_key"]).sum()))

def category(row):
    p, c = str(row.get("protein_name") or ""), str(row.get("cdna_name") or "")
    if "del" in p or "del" in c: return "deletion/indel"
    if "X" in p or "Ter" in p:   return "nonsense (stop-gain)"
    if "ins" in c or "dup" in c: return "insertion/dup"
    if ("+" in c) or ("-" in c and "c." in c): return "splice/intronic"
    if "=" in p: return "synonymous"
    return "other/complex"

nokey = cf[~cf["has_key"]]
print("\nWhat the NON-missense (no-key) variants actually are:")
print(nokey.apply(category, axis=1).value_counts().to_string())

with missense key: 780 | without: 1317

What the NON-missense (no-key) variants actually are:
deletion/indel          540
nonsense (stop-gain)    347
splice/intronic         291
other/complex            50
insertion/dup            48
synonymous               41


## 2 · When did CFTR2 first call each variant CF-causing?

A truth set alone does not make a benchmark defensible. **Temporal leakage** does the
damage: a variant whose disease-causing status was public years before a model was trained
is in that model's training data, so scoring it correctly demonstrates recall, not skill.
"The tool got this right" and "the tool memorised it" are the same observation unless you
know *when* the call was made. Only variants whose status changed *after* a model's cutoff
can tell them apart — which needs a per-variant date, and that is what this section builds.

### What "the date" means here

CFTR2's workbook has **no per-variant date field** — only the `Date:` row in its header.
The date used here is derived, and worth being exact about:

> **the first CFTR2 release whose determination for this variant reads `CF-causing`.**

That is measurable because every CFTR2 release carries **two** determination columns,
current and previous, and each release's previous-version column names the release before
it. Chained across the twelve published releases, they give a per-variant trajectory —
what CFTR2 called each variant, at each release, over a decade.

This is CFTR2's *own* assertion. Nothing is borrowed from ClinVar or the literature, and
nothing is invented.

### The floor, and what it does to F508del

The chain reaches back only as far as the oldest workbook's previous-version column,
**2015-08-13**. For anything already CF-causing at that point — F508del included — the
series records the call but not its date: the trajectory simply starts already CF-causing.
F508del was reported in 1989, and no amount of chaining recovers the intervening
quarter-century. cftr2.org serves no 2015-or-earlier workbook (checked), so this floor is
a property of what CFTR2 published, not of how the data is read here.

That is still enough to be useful: every tool in `toolkit.TOOL_YEAR` was released after the
floor, so a hold-out is constructible for all of them.

### Reproducibility

CFTR2 keeps **every past release** at a stable public path, linked from its *CFTR2 Variant
List History* page, so this section is fully reproducible — the build cell downloads any
workbooks you do not already have. No login, no session, and no site-agreement click is
needed for the files themselves (that gate is on the variant *search*, not the downloads).
Verified: all twelve fetch byte-identical to the locally archived copies.

Downloading is what CFTR2's terms permit ("solely for your own non-commercial use").
Republishing is not — so the workbooks and everything built from them stay in gitignored
`data/`, and this notebook prints only aggregates.

In [5]:
import cftr2_history as ch

CFTR2_HIST_CSV = DATA_DIR / "cftr2_history.csv"

if CFTR2_HIST_CSV.exists():
    print(f"already built -> {CFTR2_HIST_CSV.name} "
          f"(delete it and cftr2_history.release.json to rebuild)")
else:
    # Fetch any release workbooks not already in data/. Files you already have are left
    # alone rather than re-fetched, so a local copy is never silently replaced.
    fetched = ch.download_releases(DATA_DIR)
    print(f"workbooks: {len(fetched['already_present'])} already in data/, "
          f"{len(fetched['downloaded'])} downloaded from cftr2.org")
    if fetched["failed"]:
        raise RuntimeError(f"could not fetch: {fetched['failed']}")

    WORKBOOKS = [DATA_DIR / n for n in ch.CFTR2_RELEASE_FILES]
    hist = ch.build_history(WORKBOOKS)
    meta = hist["meta"]
    print(f"\nreleases chained: {meta['release_count']}  "
          f"({meta['releases'][0]} -> {meta['latest_release']})")
    print(f"series starts at: {meta['censoring_floor']}  "
          f"(the oldest workbook's previous-version column)")
    print(f"join key        : {meta['join_key']}   renames merged: {meta['renames_applied']}")
    print(f"\nchain verified contiguous across {len(meta['chain'])} steps -- "
          "each release's previous-version label matches the preceding file's own Date:")
    for step in meta["chain"]:
        print(f"  {step}")
    paths = ch.write_extracts(hist, DATA_DIR)
    print(f"\nwrote {paths['long'].name} ({len(hist['long']):,} variant x release rows), "
          f"{paths['summary'].name} ({len(hist['summary']):,} variants) "
          f"and {paths['release'].name}")

WORKBOOKS = [DATA_DIR / n for n in ch.CFTR2_RELEASE_FILES]

workbooks: 12 already in data/, 0 downloaded from cftr2.org



releases chained: 12  (2016-08-08 -> 2026-01-30)
series starts at: 2015-08-13  (the oldest workbook's previous-version column)
join key        : legacy_name   renames merged: 7

chain verified contiguous across 11 steps -- each release's previous-version label matches the preceding file's own Date:
  2016-08-08 -> 2017-03-17
  2017-03-17 -> 2017-12-08
  2017-12-08 -> 2018-08-31
  2018-08-31 -> 2019-03-11
  2019-03-11 -> 2020-01-10
  2020-01-10 -> 2020-07-31
  2020-07-31 -> 2021-09-24
  2021-09-24 -> 2022-04-29
  2022-04-29 -> 2023-04-07
  2023-04-07 -> 2024-09-25
  2024-09-25 -> 2026-01-30

wrote cftr2_history_long.csv (7,978 variant x release rows), cftr2_history.csv (2,102 variants) and cftr2_history.release.json


### Why the join key is the legacy name

Chaining twelve releases is a join, and the join is where this goes wrong quietly. Three
things in the source would each return a plausible wrong answer rather than an error, so
`cftr2_history.py` guards all three and **raises**:

| what changes | what it would do undetected | guard |
|---|---|---|
| **2023 nomenclature migration** — `c.1029delC` → `c.1029del`, `c.1021_1022dupTC` → `c.1021_1022dup`, complex alleles bracketed | keying on cDNA name loses variants at that step and re-dates long-known alleles to 2023 | dropout rate per release step |
| **class vocabulary drift** — `Unknown significance` → `No interpretation available` | a pure relabelling reads as a reclassification for every affected variant | both map to one normalised class; the raw string is kept |
| **rename tombstones** — `(CF-causing under new name)` | scored as a class it invents downgrades; dropped outright it orphans the successor, which then looks new | rename target parsed, old name aliased to new, trajectories merged |

The **legacy name** is the only identifier stable across all twelve releases, and it is
unique within every one of them — so it is the key. The cDNA name is not: it is the thing
the 2023 migration rewrote.

The dropout guard is the one that matters, and the more obvious "did a variant vanish and
come back?" check is **not** a substitute for it. When a key breaks, most affected variants
do not vanish and return — they vanish under the old name and reappear under a new one, so
they read as one cohort dropped and another added, and every one is silently re-dated. Only
the few that happen to revert would trip a resurrection check.

The cell below runs the build again deliberately keyed on the cDNA name, to show the guard
firing on the failure it exists to catch rather than merely asserting that it would.

In [6]:
# NEGATIVE CONTROL -- this build is SUPPOSED to fail.
# Guard messages can name variants; CFTR2 names are not republished here, so anything
# after an "(e.g. ..." is cut before printing.
def redact(err):
    return str(err).split("(e.g.")[0].strip()

try:
    ch.build_history(WORKBOOKS, key_by="cdna_name")
    print("NO GUARD FIRED -- the negative control did not fail, which means the guard "
          "is no longer protecting anything. Do not trust the dates.")
except ch.HistoryError as e:
    print("guard fired as intended, keyed on cDNA name:\n")
    print(" ", redact(e))

# ...and the same build on the real key must succeed, or the comparison proves nothing.
ch.build_history(WORKBOOKS)
print("\nsame twelve workbooks keyed on legacy name: builds cleanly.")

guard fired as intended, keyed on cDNA name:

  39 of 400 variants present at 2018-08-31 are absent at 2019-03-11 (limit 5). CFTR2 does not retire variants in bulk, so the join key has almost certainly broken: those variants are still in the list under a changed name and every one of them would be re-dated to 2019-03-11. Check the key column against that release's nomenclature.



same twelve workbooks keyed on legacy name: builds cleanly.


### The output: one row per variant per release

The build writes `data/cftr2_history_long.csv` — the actual result of this section, keyed
on **(variant, release date)** with CFTR2's determination at that release:

| column | |
|---|---|
| `variant_key` | CFTR2's legacy name — stable across all twelve releases |
| `release_date` | which release this row describes |
| `determination_raw` | CFTR2's exact string for that variant at that release |
| `determination` | the same, normalised (`cf_causing` / `varying` / `non_cf_causing` / `no_interpretation`) |

Both the raw string and the normalised form are kept. The normalisation only merges
`Unknown significance` and `No interpretation available`, which are the same class renamed
between releases; keeping the raw column means that judgement can be checked rather than
taken on trust.

This is the primitive everything else is derived from — a variant's whole history is the
rows sharing its key, read in date order. `data/cftr2_history.csv` sits beside it with
per-variant summaries (first CF-causing release, number of changes) for the hold-out below.

**Neither file is committed.** CFTR2's terms forbid republishing any portion of the
Content, and a per-variant table of its determinations is exactly that. Both are built into
gitignored `data/` and rebuilt by anyone running this notebook; only counts appear below.

In [7]:
long = pd.read_csv(DATA_DIR / "cftr2_history_long.csv", parse_dates=["release_date"])
summary = tk.load_cftr2_history()

# 13 dates, not 12: the oldest workbook's previous-version column is a real observation of
# an earlier state, so 2015-08-13 is carried as a row even though no workbook bears it.
print(f"table shape      : {len(long):,} rows x {long.shape[1]} columns "
      "(one row per variant per release)")
print(f"dates covered    : {long['release_date'].nunique()} "
      f"= 12 published releases + the {long['release_date'].min().date()} state")
print(f"distinct variants: {long['variant_key'].nunique():,} "
      f"({int(summary['cftr2_in_current_release'].sum()):,} in the current 2026 release, "
      f"{int((~summary['cftr2_in_current_release']).sum())} since retired from the list)")


def trajectory(states):
    """Summarise one variant's determinations, read in release order."""
    v = list(states)
    ever = "cf_causing" in v
    i = v.index("cf_causing") if ever else None
    return pd.Series({
        "changed":   len(set(v)) > 1,
        "ever_cf":   ever,
        # started as something else and later became CF-causing
        "became_cf": ever and v[0] != "cf_causing",
        # was CF-causing and at some LATER release was not (a round trip counts)
        "withdrew":  ever and any(x != "cf_causing" for x in v[i + 1:]),
    })


# Comparing the NORMALISED determination is what keeps CFTR2's rename of "Unknown
# significance" to "No interpretation available" from counting as hundreds of changes.
t = (long.sort_values(["variant_key", "release_date"])
         .groupby("variant_key")["determination"].apply(trajectory).unstack())

print(f"\ntotal variants tracked                  : {len(t):,}")
print(f"ever called CF-causing                  : {int(t['ever_cf'].sum()):,}")
print(f"changed determination at least once     : {int(t['changed'].sum())} "
      f"({t['changed'].mean():.2%})")
print(f"  became CF-causing, having started else: {int(t['became_cf'].sum())}")
print(f"  were CF-causing and later were not    : {int(t['withdrew'].sum())}")


table shape      : 7,978 rows x 6 columns (one row per variant per release)
dates covered    : 13 = 12 published releases + the 2015-08-13 state
distinct variants: 2,102 (2,092 in the current 2026 release, 10 since retired from the list)



total variants tracked                  : 2,102
ever called CF-causing                  : 1,255
changed determination at least once     : 23 (1.09%)
  became CF-causing, having started else: 4
  were CF-causing and later were not    : 1


### What this buys: a hold-out per tool

For each predictor, the variants CFTR2 first called CF-causing **after** that tool was
released could not have been in its training data as CF-causing. Those are the ones where a
correct score is evidence of skill rather than recall.

`TOOL_YEAR` records release years, so the comparison below is by year and deliberately
conservative: a variant counts only if its first CF-causing release falls in a year
*strictly after* the tool's. Same-year cases are excluded rather than guessed at.

Note which row matters most. `LABEL_SUPERVISED` marks the tools trained directly on curated
clinical labels — leakage is a first-order problem for those and a second-order one for the
rest, which never saw clinical labels at all and can only leak through the literature that
informed them.

In [8]:
# Variants CFTR2 first called CF-causing AFTER a tool shipped could not have been in that
# tool's training data as CF-causing. TOOL_YEAR is year-granular, so this is deliberately
# conservative: same-year cases are excluded rather than guessed at.
current = summary[summary["cftr2_in_current_release"]]        # the benchmark set as it stands
first_cf = pd.to_datetime(current["cftr2_first_cf_causing"], errors="coerce")
# Anything already CF-causing at the 2015-08-13 floor has no date, only that lower bound,
# so it can never count toward a hold-out -- which is the safe direction.
datable = first_cf[first_cf > pd.Timestamp("2015-08-13")]

holdout = pd.DataFrame(
    [{"tool": t,
      "released": yr,
      "label_supervised": tk.LABEL_SUPERVISED[t],
      "cf_causing_after_release": int((datable.dt.year > yr).sum())}
     for t, yr in sorted(tk.TOOL_YEAR.items(), key=lambda kv: kv[1])])
print(holdout.to_string(index=False))

never_cf = int((current["cftr2_current_determination"] != "cf_causing").sum())
print()
print(f"candidate negatives available to every tool: {never_cf:,} variants "
      "CFTR2 does not currently call CF-causing")


         tool  released  label_supervised  cf_causing_after_release
        REVEL      2016              True                       977
    PrimateAI      2018             False                       912
     SpliceAI      2019             False                       901
          EVE      2021             False                       865
     Pangolin      2022             False                       847
AlphaMissense      2023             False                       531
        ESM1b      2023             False                       531
         CADD      2024             False                       164

candidate negatives available to every tool: 847 variants CFTR2 does not currently call CF-causing


### What this date can and cannot support

**It can** separate recall from skill for any tool released after the series begins, and it
says which variants to use: score only the ones CFTR2 first called CF-causing after the
tool shipped, against the variants it does not call CF-causing as negatives.

**It cannot** tell you when a variant was *discovered*, or when anyone first called it
disease-causing. It measures one thing — when **CFTR2** adopted the call. Three gaps follow
from that, and none are closed by more careful chaining:

- **Everything at the 2015-08-13 floor is one bucket.** Those variants are ordered relative
  to nothing. F508del and a variant first described in 2014 are indistinguishable.
- **Resolution is the release cadence**, 6–18 months. A variant dated to a release became
  CF-causing at some point since the previous one.
- **CFTR2 is a follower, not a first mover.** A variant is typically called CF-causing in
  the literature and in ClinVar before CFTR2 adopts it, so the date is an **upper bound** on
  when the label became public. Used for a hold-out it errs the safe way — it can wrongly
  exclude a variant as leaked, but it will not wrongly admit one.

That last point is the honest limit on the whole section. This dates CFTR2's adoption of a
call, which is a defensible and conservative proxy for when the label was learnable — not
the date the knowledge first existed.

## Key takeaways

1. **CFTR2** calls combine **patient data + functional assays** — a functional axis ClinVar largely lacks. It is not an independent gold standard; see [`../tools/05_revel.ipynb`](../tools/05_revel.ipynb) §2.
2. CFTR2's own header counts **2,092** variants (the built extract has 2,097 rows because five trailing footnote and copyright lines sit in the variant-name column). **780** have a 1-letter missense key, 779 of which join to AlphaMissense; the rest are non-missense — mostly deletions and nonsense, not splice — and join by `cdna_name` / genomic coordinates.
3. **Version:** every release stays downloadable from cftr2.org at `sites/default/files/CFTR2_<DDMonthYYYY>.xlsx`, so a run can be pinned to a specific release. The build cell records that release's own header date as `cftr2_release`.
4. **§2 reconstructs what CFTR2 called each variant at each of its twelve releases**, 2015-08-13 → 2026-01-30, into `data/cftr2_history_long.csv` (gitignored — a per-variant table of CFTR2 determinations is exactly what its terms forbid republishing). That table is what a training-cutoff hold-out is built from.
5. Determinations are **mostly stable**: 23 of 2,102 tracked variants ever changed class, 4 became CF-causing having started as something else, and 1 was CF-causing and later was not. The dates are an **upper bound** on when a label became public — CFTR2 adopts calls after the literature does.

